appears to be fully working, at least for moving, which is what matters...

Next, will try to make the reply human readable

In [30]:
import socket
import time
import xml.etree.ElementTree as ET

class GilsonSession:
    """
    A class to manage a session with a Gilson Ethernet-controlled device.
    Handles connection, command formatting, sending, and response parsing.
    """

    def __init__(self, ip, admin_port=50185):
        """
        Initialize the session and connect to the Gilson controller.

        :param ip: IP address of the Gilson device.
        :param admin_port: Port to initiate the session (default is 50185).
        """
        self.ip = ip
        self.admin_port = admin_port
        self.session_port = None
        self.session_socket = None
        self.sequence_number = 40  # Start from 40 so first increment is 41
        self._connect()
        self.send_admin_command()  # Recommendation 2: send admin command on init

    def _connect(self):
        """
        Establish the initial admin connection and obtain a session port.
        Then open a persistent session socket.
        """
        message = "<Gilson><GilsonConnect>Gilson Ethernet Utility</GilsonConnect></Gilson>"

        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(5)
            s.connect((self.ip, self.admin_port))
            s.sendall(message.encode('ascii'))
            response = s.recv(4096).decode('ascii')

        if "<ParameterName>ResponseValue</ParameterName>" in response:
            port = int(response.split("<ParameterValue>")[2].split("</ParameterValue>")[0])
            self.session_port = port

            self.session_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.session_socket.settimeout(5)
            self.session_socket.connect((self.ip, self.session_port))
        else:
            raise RuntimeError("Failed to retrieve assigned session port from Gilson.")

    def send_raw_command(self, xml_payload):
        """
        Send raw XML to the device and read the response.

        :param xml_payload: XML-formatted string command.
        :return: Full XML response string.
        """
        try:
            self.session_socket.sendall(xml_payload.encode('ascii'))
            time.sleep(1.0)  # Increased delay to better capture full response
            buffer = b""
            while True:
                try:
                    part = self.session_socket.recv(8192)
                    if not part:
                        break
                    buffer += part
                    if b"</Message>" in part:
                        break
                except socket.timeout:
                    break
            response = buffer.decode('ascii', errors='ignore')
            print(f"Full XML response:\n{response}\n")
            return response
        except Exception as e:
            print("Error sending raw command:", e)
            return None

    def _get_next_sequence_number(self):
        """
        Increment and return the next sequence number.
        """
        self.sequence_number += 1
        return self.sequence_number

    def make_command(self, command_name, device="GX-27x", device_id=35, sequence_number=None, command_type="Local", parameters=None):
        """
        Format a full XML command.

        :param command_name: Name of the command (e.g., 'Identify').
        :param device: Name of the device (default 'GX-27x').
        :param device_id: Device ID (default 35).
        :param sequence_number: Sequence number (auto-incremented if None).
        :param command_type: 'Local' or 'Immediate'.
        :param parameters: Optional dictionary of XML parameters.
        :return: XML string.
        """
        if sequence_number is None:
            sequence_number = self._get_next_sequence_number()

        param_xml = ""
        if parameters:
            param_xml = "<Parameters>"
            for key, value in parameters.items():
                param_xml += f"<Parameter><ParameterName>{key}</ParameterName><ParameterValue>{value}</ParameterValue></Parameter>"
            param_xml += "</Parameters>"

        xml = f"""
<Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>{command_type}</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>{device}</DeviceName>
            <DeviceId>{device_id}</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>{sequence_number}</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>{command_name}</CommandName>
            {param_xml}
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson>"""
        return xml.strip()

    def send_command(self, command_name, device="GX-27x", device_id=35, sequence_number=None, parameters=None):
        """
        Send a 'Local' command to the device.
        """
        xml = self.make_command(command_name, device, device_id, sequence_number, command_type="Local", parameters=parameters)
        return self.send_raw_command(xml)

    def send_immediate_command(self, command_name, device="GX-27x", device_id=35, sequence_number=None, parameters=None):
        """
        Send an 'Immediate' command to the device.
        """
        xml = self.make_command(command_name, device, device_id, sequence_number, command_type="Immediate", parameters=parameters)
        return self.send_raw_command(xml)

    def send_admin_command(self):
        """
        Send the 'Admin' command using standard Local format with DeviceId=-1 and SequenceNumber=0.
        """
        xml = """
<Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Local</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>System</DeviceName>
            <DeviceId>-1</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>0</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Admin</CommandName>
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson>"""
        return self.send_raw_command(xml.strip())

    def move_x(self, position):
        """
        Move the autosampler X-axis to a specific position.

        :param position: The target X-axis position as a string.
        """
        parameters = {"X Position": position}
        return self.send_command("Move X", parameters=parameters)

    def move_y(self, position):
        """
        Move the autosampler Y-axis to a specific position.

        :param position: The target Y-axis position as a string.
        """
        parameters = {"Y Position": position}
        return self.send_command("Move Y", parameters=parameters)

    def move_z(self, position):
        """
        Move the autosampler Z-axis to a specific position.

        :param position: The target Z-axis position as a string.
        """
        parameters = {"Z Position": position}
        return self.send_command("Move Z", parameters=parameters)

    def move_xy(self, x_position, y_position):
        """
        Move the autosampler to specified X and Y positions.

        :param x_position: The target X-axis position.
        :param y_position: The target Y-axis position.
        """
        parameters = {
            "X Position": x_position,
            "Y Position": y_position
        }
        return self.send_command("Move XY", parameters=parameters)

    def close(self):
        """
        Close the session socket.
        """
        if self.session_socket:
            self.session_socket.close()
            self.session_socket = None


In [31]:
g = GilsonSession("192.168.1.101")

Full XML response:
<Message MessageId="1665" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Saf

In [32]:
g.send_admin_command()

Full XML response:
<Message MessageId="1668" BlockCount="1" BlockId="0"><Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Local</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>System</DeviceName>
            <DeviceId>-1</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>0</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Admin</CommandName>
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson></Message><Message MessageId="1669" BlockCount="1" BlockId="0"><Gilson><Response><SequenceNumber>0</SequenceNumber><CommandName>Admin</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue>Success</ParameterValue></Parameter></Paramete

'<Message MessageId="1668" BlockCount="1" BlockId="0"><Gilson>\n  <CommandData>\n    <Commands>\n      <Command>\n        <CommandType>Local</CommandType>\n        <CommandInfo>\n          <InstrumentInfo>\n            <DeviceName>System</DeviceName>\n            <DeviceId>-1</DeviceId>\n          </InstrumentInfo>\n          <SequenceNumber>0</SequenceNumber>\n          <Synchronize>True</Synchronize>\n          <Selected>False</Selected>\n          <CommandXML>\n            <CommandName>Admin</CommandName>\n          </CommandXML>\n        </CommandInfo>\n      </Command>\n    </Commands>\n  </CommandData>\n</Gilson></Message><Message MessageId="1669" BlockCount="1" BlockId="0"><Gilson><Response><SequenceNumber>0</SequenceNumber><CommandName>Admin</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue>Success</ParameterValue></Parameter></Parame

In [16]:
g.send_command("Identify")

Full XML response:
<Message MessageId="1455" BlockCount="1" BlockId="0"><Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Local</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>GX-27x</DeviceName>
            <DeviceId>35</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>41</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Identify</CommandName>
            
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson></Message><Message MessageId="1456" BlockCount="1" BlockId="0"><Gilson><Response><SequenceNumber>41</SequenceNumber><CommandName>Identify</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><Return

'<Message MessageId="1455" BlockCount="1" BlockId="0"><Gilson>\n  <CommandData>\n    <Commands>\n      <Command>\n        <CommandType>Local</CommandType>\n        <CommandInfo>\n          <InstrumentInfo>\n            <DeviceName>GX-27x</DeviceName>\n            <DeviceId>35</DeviceId>\n          </InstrumentInfo>\n          <SequenceNumber>41</SequenceNumber>\n          <Synchronize>True</Synchronize>\n          <Selected>False</Selected>\n          <CommandXML>\n            <CommandName>Identify</CommandName>\n            \n          </CommandXML>\n        </CommandInfo>\n      </Command>\n    </Commands>\n  </CommandData>\n</Gilson></Message><Message MessageId="1456" BlockCount="1" BlockId="0"><Gilson><Response><SequenceNumber>41</SequenceNumber><CommandName>Identify</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><Ret

In [17]:
g.send_immediate_command("Get XY Position")

Full XML response:
<Message MessageId="1457" BlockCount="1" BlockId="0"><Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Immediate</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>GX-27x</DeviceName>
            <DeviceId>35</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>42</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Get XY Position</CommandName>
            
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson></Message><Message MessageId="1458" BlockCount="1" BlockId="0"><Gilson><Response><SequenceNumber>42</SequenceNumber><CommandName>Get XY Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><Return

'<Message MessageId="1457" BlockCount="1" BlockId="0"><Gilson>\n  <CommandData>\n    <Commands>\n      <Command>\n        <CommandType>Immediate</CommandType>\n        <CommandInfo>\n          <InstrumentInfo>\n            <DeviceName>GX-27x</DeviceName>\n            <DeviceId>35</DeviceId>\n          </InstrumentInfo>\n          <SequenceNumber>42</SequenceNumber>\n          <Synchronize>True</Synchronize>\n          <Selected>False</Selected>\n          <CommandXML>\n            <CommandName>Get XY Position</CommandName>\n            \n          </CommandXML>\n        </CommandInfo>\n      </Command>\n    </Commands>\n  </CommandData>\n</Gilson></Message><Message MessageId="1458" BlockCount="1" BlockId="0"><Gilson><Response><SequenceNumber>42</SequenceNumber><CommandName>Get XY Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><Ret

In [36]:
g.send_immediate_command("Identify")
g.send_immediate_command("Get Z Position")

Full XML response:
<Message MessageId="1712" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Saf

Full XML response:
ameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command><Command><CommandName>Get Error</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Error</ParameterName><ParameterValue>0 No error</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command><Command><CommandName>Get XY Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>X Position</ParameterName><ParameterValue>3.8</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Y Position</</Message><Message MessageId="1713" Block

'ameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command><Command><CommandName>Get Error</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Error</ParameterName><ParameterValue>0 No error</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command><Command><CommandName>Get XY Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>X Position</ParameterName><ParameterValue>3.8</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Y Position</</Message><Message MessageId="1713" BlockCount="2" BlockId=

In [40]:
g.send_immediate_command("Home")
g.move_x("100")
g.move_y("100")
g.send_immediate_command("Home")
g.move_xy("100","100")
g.move_z("100")
g.send_immediate_command("Home")

Full XML response:
>
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson></Message><Message MessageId="1779" BlockCount="1" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get XY Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>X Position</ParameterName><ParameterValue>4.572</ParameterValue></ReturnParameter><ReturnParameter><ParameterN

Full XML response:
vent System</DeviceName><DeviceId>8</DeviceId><Connected>False</Connected></Device><Device><DeviceName>GX Direct Inject</DeviceName><DeviceId>9</DeviceId><Connected>False</Connected></Device><Device><DeviceName>406 Single Syringe Pump</DeviceName><DeviceId>7</DeviceId><Connected>False</Connected></Device></Devices></Status></Gilson></Message><Message MessageId="1785" BlockCount="1" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get XY Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Para

Full XML response:
arameters><ReturnParameter><ParameterName>Solenoid 1</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Solenoid 2</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command><Command><CommandName>Get 24V Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>24V Contact 1</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>24V Contact 2</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command><Command><CommandName>Get Diverter Valve</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><P

Full XML response:
evice></Devices></Status></Gilson></Message><Message MessageId="1791" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><ParameterValue>Off</ParameterValue></ReturnP

Full XML response:
alve</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command><Command><CommandName>Get Error</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Error</ParameterName><ParameterValue>0 No error</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command><Command><CommandName>Get XY Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>X Position</ParameterName><ParameterValue>3.8</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Y Positio

Full XML response:
/Parameters></Command><Command><CommandName>Get Output Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact 1</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact 2</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command><Command><CommandName>Get Solenoid Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Solenoid 1</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Solenoid

Full XML response:
lue></Parameter></Parameters></Command><Command><CommandName>Get Z Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Z Position</ParameterName><ParameterValue>175</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command></Commands></Device><Device><DeviceName>GX Solvent System</DeviceName><DeviceId>8</DeviceId><Connected>False</Connected></Device><Device><DeviceName>GX Direct Inject</DeviceName><DeviceId>9</DeviceId><Connected>False</Connected></Device><Device><DeviceName>406 Single Syringe Pump</DeviceName><DeviceId>7</DeviceId><Connected>False</Connected></Device></Devices></Status></Gilson></Message><Message MessageId="1796" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAd

'lue></Parameter></Parameters></Command><Command><CommandName>Get Z Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Z Position</ParameterName><ParameterValue>175</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Command></Commands></Device><Device><DeviceName>GX Solvent System</DeviceName><DeviceId>8</DeviceId><Connected>False</Connected></Device><Device><DeviceName>GX Direct Inject</DeviceName><DeviceId>9</DeviceId><Connected>False</Connected></Device><Device><DeviceName>406 Single Syringe Pump</DeviceName><DeviceId>7</DeviceId><Connected>False</Connected></Device></Devices></Status></Gilson></Message><Message MessageId="1796" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>

In [27]:
g.send_immediate_command("Home")

Full XML response:
<Message MessageId="1597" BlockCount="1" BlockId="0"><Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Immediate</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>GX-27x</DeviceName>
            <DeviceId>35</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>41</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Home</CommandName>
            
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson></Message><Message MessageId="1598" BlockCount="1" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceNa

'<Message MessageId="1597" BlockCount="1" BlockId="0"><Gilson>\n  <CommandData>\n    <Commands>\n      <Command>\n        <CommandType>Immediate</CommandType>\n        <CommandInfo>\n          <InstrumentInfo>\n            <DeviceName>GX-27x</DeviceName>\n            <DeviceId>35</DeviceId>\n          </InstrumentInfo>\n          <SequenceNumber>41</SequenceNumber>\n          <Synchronize>True</Synchronize>\n          <Selected>False</Selected>\n          <CommandXML>\n            <CommandName>Home</CommandName>\n            \n          </CommandXML>\n        </CommandInfo>\n      </Command>\n    </Commands>\n  </CommandData>\n</Gilson></Message><Message MessageId="1598" BlockCount="1" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><Devic

In [7]:
#g.close()